# Financial fraud detector for fintech

THe goal is for a transaction data to be provided and the fine-tuned model will predict if it is a fraudulent or not, with Confidence level and reference to short helpful reason why it arrived at the conclusion

- Use data set from Hugginface to fine-tune a model
- Transaction data will be provided to both base model and the fine-tuned model
- Chat with the fine-tuned model from gradio interface.

Available to run on Colab via https://colab.research.google.com/drive/11jdEmwowviYMMAr_To7h5lZHviBivqnA?usp=sharing

## 1. Imports and Setup

In [ ]:
# Colab: uncomment to install (run once)
!pip install -q transformers peft accelerate datasets trl torch

In [ ]:
from google.colab import userdata

In [ ]:
# Colab: uncomment to install (run once)
# !pip install -q transformers peft accelerate datasets trl torch

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import json
from pathlib import Path

from dotenv import load_dotenv
from datasets import load_dataset, Dataset
from huggingface_hub import login
from openai import OpenAI
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
import gradio as gr
import torch

load_dotenv(override=True)
OPENROUTER_URL = "https://openrouter.ai/api/v1"
HF_TOKEN = userdata.get("HF_TOKEN")
OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY") or userdata.get("OPENAI_API_KEY")

if HF_TOKEN:
    login(HF_TOKEN, add_to_git_credential=True)
if OPENROUTER_API_KEY:
    openrouter_client = OpenAI(api_key=OPENROUTER_API_KEY, base_url=OPENROUTER_URL)
    print("OpenRouter client initialized")
else:
    openrouter_client = None
    print("OPENROUTER_API_KEY or OPENAI_API_KEY not set - base model inference will fail")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


OpenAI client initialized


## 2. Load Dataset from HuggingFace

Load `qppd/bank-transaction-fraud` (3,000 rows). Sample 50 train, 20 val (stratified), reserve ~30 for test.

In [9]:
# Constants - minimal subset for ~5-10 min fine-tuning on Colab
TRAIN_SIZE = 50
VAL_SIZE = 20
TEST_SIZE = 30
RANDOM_STATE = 42
MODEL_NAME = "unsloth/Llama-3.2-1B-Instruct"
BASE_MODEL_OPENROUTER = "meta-llama/llama-3.2-1b-instruct"
LORA_SAVE_PATH = "./fraud_detector_lora"

In [10]:
# Load qppd/bank-transaction-fraud (3k rows, ~60KB)
dataset = load_dataset("qppd/bank-transaction-fraud", split="train")
df = dataset.to_pandas()

# qppd columns: transactions, locations, total_amount, limit, result
# result: 0=legitimate, 1 or 2=fraudulent
df["is_fraud"] = (df["result"] != 0).astype(int)

# Stratified split: 50 train, 20 val, 30 test
total_needed = TRAIN_SIZE + VAL_SIZE + TEST_SIZE
fraud_pct = df["is_fraud"].mean()
print(f"Dataset: {len(df)} rows, fraud rate: {fraud_pct:.1%}")

subset, _ = train_test_split(df, train_size=total_needed, stratify=df["is_fraud"], random_state=RANDOM_STATE)
train_df, rest = train_test_split(subset, train_size=TRAIN_SIZE, stratify=subset["is_fraud"], random_state=RANDOM_STATE)
val_df, test_df = train_test_split(rest, train_size=VAL_SIZE, stratify=rest["is_fraud"], random_state=RANDOM_STATE)

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")
print(f"Train fraud: {train_df['is_fraud'].sum()}, Val fraud: {val_df['is_fraud'].sum()}")

Dataset: 3000 rows, fraud rate: 62.8%
Train: 50, Val: 20, Test: 30
Train fraud: 32, Val fraud: 12


## 3. Convert to Text Format

Create concise human-readable transaction summaries (5-8 fields) for LLM prompts.

In [11]:
def transaction_to_text(row):
    """Convert a transaction row to a concise natural-language summary."""
    return (
        f"Transaction: {row['total_amount']:,.2f} amount, {row['transactions']} transactions, "
        f"{row['locations']} locations. Limit: {row['limit']:,}."
    )

# Store formatted text and labels for each split
train_data = [
    {"text": transaction_to_text(row), "is_fraud": bool(row["is_fraud"])}
    for _, row in train_df.iterrows()
]
val_data = [
    {"text": transaction_to_text(row), "is_fraud": bool(row["is_fraud"])}
    for _, row in val_df.iterrows()
]
test_data = [
    {"text": transaction_to_text(row), "is_fraud": bool(row["is_fraud"])}
    for _, row in test_df.iterrows()
]

print("Sample train entry:")
print(train_data[0])

Sample train entry:
{'text': 'Transaction: 4,173.70 amount, 1.0 transactions, 1.0 locations. Limit: 50,000.0.', 'is_fraud': False}


## 4. Format for SFT and Create Dataset

Convert fraud data to instruction format for Llama (chat template). Create HuggingFace Dataset for SFTTrainer.

In [12]:
SYSTEM_PROMPT = "Analyze this bank transaction. Reply with: Fraudulent or Legitimate, Confidence: X%, Reason: short explanation."

def format_for_sft(item):
    """Convert fraud item to chat format for Llama SFT (### System/User/Assistant)."""
    label = "Fraudulent" if item["is_fraud"] else "Legitimate"
    conf = 85 if item["is_fraud"] else 92
    reason = "Suspicious pattern." if item["is_fraud"] else "Normal transaction pattern."
    assistant = f"{label}. Confidence: {conf}%. Reason: {reason}"
    text = (
        f"### System:\n{SYSTEM_PROMPT}\n\n### User:\nIs this transaction fraudulent?\n\n{item['text']}\n\n### Assistant:\n{assistant}"
    )
    return text

train_texts = [format_for_sft(d) for d in train_data]
val_texts = [format_for_sft(d) for d in val_data]
train_dataset = Dataset.from_dict({"text": train_texts})
val_dataset = Dataset.from_dict({"text": val_texts})

print("Example training text:")
print(train_texts[0][:300] + "...")

{
  "messages": [
    {
      "role": "user",
      "content": "Analyze this bank transaction. Reply with: Fraudulent or Legitimate, Confidence: X%, Reason: short explanation.\n\nIs this transaction fraudulent?\n\nTransaction: 4,173.70 amount, 1.0 transactions, 1.0 locations. Limit: 50,000.0."
    },
    {
      "role": "assistant",
      "content": "Legitimate. Confidence: 92%. Reason: Normal transaction pattern."
    }
  ]
}


In [13]:
print(f"Train dataset: {len(train_dataset)} samples")
print(f"Val dataset: {len(val_dataset)} samples")

Wrote 50 train, 20 val to jsonl/


## 5. Load Model and Fine-tune with LoRA

Load Llama 3.2 1B, add LoRA adapters, train with SFTTrainer (~5-10 min on Colab).

In [14]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

device_map = "auto" if torch.cuda.is_available() else None
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16 if (torch.cuda.is_available() or torch.backends.mps.is_available()) else torch.float32,
    device_map=device_map,
)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
model.tokenizer = tokenizer
print("Model loaded with LoRA adapters")

APIStatusError: Error code: 405

## 5b. Training Configuration and Train

In [ ]:
training_args = TrainingArguments(
    output_dir=LORA_SAVE_PATH,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    max_steps=80,
    learning_rate=2e-4,
    bf16=torch.cuda.is_available() or torch.backends.mps.is_available(),
    logging_steps=10,
    save_strategy="steps",
    save_steps=40,
    save_total_limit=2,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    args=training_args,
)
print("Trainer initialized")

In [ ]:
trainer.train()
model.save_pretrained(LORA_SAVE_PATH)
tokenizer.save_pretrained(LORA_SAVE_PATH)
print(f"Fine-tuned model saved to {LORA_SAVE_PATH}/")

## 6. Load Fine-tuned Model and Inference

Load saved LoRA adapters for inference. Base model via OpenRouter, fine-tuned model locally.

In [ ]:
from peft import PeftModel

# Load base model and LoRA adapters for inference
ft_base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)
ft_model = PeftModel.from_pretrained(ft_base_model, LORA_SAVE_PATH)
ft_model.eval()
ft_tokenizer = AutoTokenizer.from_pretrained(LORA_SAVE_PATH)
print("Fine-tuned model loaded for inference")

In [ ]:
def predict_finetuned(transaction_text):
    """Run fine-tuned model locally. Returns raw response string."""
    if not transaction_text:
        return "No input."
    prompt = f"### System:\n{SYSTEM_PROMPT}\n\n### User:\nIs this transaction fraudulent?\n\n{transaction_text}\n\n### Assistant:\n"
    inputs = ft_tokenizer(prompt, return_tensors="pt").to(ft_model.device)
    with torch.no_grad():
        out = ft_model.generate(**inputs, max_new_tokens=80, do_sample=False, pad_token_id=ft_tokenizer.eos_token_id)
    reply = ft_tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    return reply or "No response generated."


def predict_base_openrouter(transaction_text):
    """Call base model via OpenRouter. Returns raw response string."""
    if not openrouter_client:
        return "OpenRouter not configured. Set OPENROUTER_API_KEY or OPENAI_API_KEY."
    user_msg = f"Is this transaction fraudulent?\n\n{transaction_text}" if transaction_text else "No input."
    resp = openrouter_client.chat.completions.create(
        model=BASE_MODEL_OPENROUTER,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ],
        max_tokens=80,
        temperature=0,
    )
    return resp.choices[0].message.content.strip()


def compare_models(transaction_text):
    """Call both fine-tuned (local) and base (OpenRouter). Returns (finetuned_response, base_response)."""
    ft_response = predict_finetuned(transaction_text)
    base_response = predict_base_openrouter(transaction_text)
    return ft_response, base_response

## 7. Gradio Interface

Split layout: fine-tuned on left, base on right, chat input above.

In [ ]:
with gr.Blocks(title="Financial Fraud Detector") as demo:
    gr.Markdown("# Financial Fraud Detector")
    inp = gr.Textbox(
        label="Transaction data",
        placeholder="Paste transaction details (e.g. Transaction: 50,000 amount, 3 transactions, 2 locations. Limit: 50,000.)",
        lines=5,
    )
    with gr.Row():
        out_finetuned = gr.Textbox(label="Fine-tuned model (left)", lines=6)
        out_base = gr.Textbox(label="Base model (right)", lines=6)
    gr.Button("Analyze").click(fn=compare_models, inputs=inp, outputs=[out_finetuned, out_base])

demo.launch()